# 02 Machine Learning Models

This notebook trains and tunes Random Forest, Support Vector Machine, k-Nearest Neighbors, and XGBoost models for district-level multiclass UHI classification.

**Project:** Comparative Study of Machine Learning Models for District-Level Urban Heat Island Classification in Taiwan

**Author:** Zalfa Afifah Zahra

## Model training and hyperparameter tuning

Train models on the original and SMOTE-balanced datasets using GridSearchCV and five-fold cross-validation.

In [ ]:
# =========================================================
# IMPORTS
# =========================================================

import pandas as pd
import joblib

from sklearn.model_selection import GridSearchCV

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier

# =========================================================
# TRAINING DATASETS
# Assumes these already exist:
# X_train
# y_train
# X_train_smote
# y_train_smote
# X_test
# y_test
# =========================================================

datasets = {

    "Original": (
        X_train,
        y_train
    ),

    "SMOTE": (
        X_train_smote,
        y_train_smote
    )
}

# =========================================================
# RANDOM FOREST
# =========================================================

rf_model = RandomForestClassifier(

    random_state=42,
    n_jobs=-1

)

rf_params = {

    "n_estimators":[200,300,500],

    "max_depth":[
        None,
        10,
        20,
        30
    ],

    "min_samples_split":[
        2,
        5,
        10
    ],

    "min_samples_leaf":[
        1,
        2,
        4
    ],

    "max_features":[
        "sqrt",
        "log2"
    ]
}

# =========================================================
# SVM
# =========================================================

svm_model = Pipeline([

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        SVC(

            kernel="rbf",

            probability=True,

            random_state=42
        )
    )

])

svm_params = {

    "model__C":[
        0.1,
        1,
        10,
        100
    ],

    "model__gamma":[
        "scale",
        0.001,
        0.01,
        0.1,
        1
    ]
}

# =========================================================
# KNN
# =========================================================

knn_model = Pipeline([

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        KNeighborsClassifier()
    )

])

knn_params = {

    "model__n_neighbors":[
        3,
        5,
        7,
        9,
        11,
        15
    ],

    "model__weights":[
        "uniform",
        "distance"
    ],

    "model__metric":[
        "euclidean",
        "manhattan",
        "minkowski"
    ]
}

# =========================================================
# XGBOOST
# =========================================================

xgb_model = XGBClassifier(

    objective="multi:softprob",

    num_class=5,

    eval_metric="mlogloss",

    random_state=42,

    tree_method="hist"

)

xgb_params = {

    "n_estimators":[
        200,
        300,
        500
    ],

    "max_depth":[
        3,
        5,
        7,
        10
    ],

    "learning_rate":[
        0.01,
        0.05,
        0.1
    ],

    "subsample":[
        0.8,
        1.0
    ],

    "colsample_bytree":[
        0.8,
        1.0
    ],

    "min_child_weight":[
        1,
        3,
        5
    ]
}

# =========================================================
# MODEL COLLECTION
# =========================================================

models = {

    "Random Forest": (
        rf_model,
        rf_params
    ),

    "SVM": (
        svm_model,
        svm_params
    ),

    "kNN": (
        knn_model,
        knn_params
    ),

    "XGBoost": (
        xgb_model,
        xgb_params
    )
}

# =========================================================
# STORAGE
# =========================================================

best_models = {}

best_parameters = []

results = []

# =========================================================
# TRAIN ALL MODELS
# =========================================================

for dataset_name, (X_tr, y_tr) in datasets.items():

    print("\n")
    print("="*70)
    print(f"TRAINING USING {dataset_name.upper()} DATA")
    print("="*70)

    for model_name, (model, params) in models.items():

        print(f"\nTUNING {model_name}")
        print("-"*50)

        grid = GridSearchCV(

            estimator=model,

            param_grid=params,

            cv=5,

            scoring="f1_macro",

            n_jobs=-1,

            verbose=1

        )

        grid.fit(
            X_tr,
            y_tr
        )

        model_key = (
            f"{model_name}_{dataset_name}"
        )

        best_models[model_key] = (
            grid.best_estimator_
        )

        # =================================================
        # SAVE PARAMETERS
        # =================================================

        best_parameters.append({

            "Dataset":
                dataset_name,

            "Model":
                model_name,

            "Best_F1_CV":
                round(
                    grid.best_score_,
                    4
                ),

            "Best_Parameters":
                str(
                    grid.best_params_
                )

        })

        print(
            "Best CV F1:",
            round(
                grid.best_score_,
                4
            )
        )

        print(
            grid.best_params_
        )

        # =================================================
        # TEST EVALUATION
        # =================================================

        y_pred = grid.best_estimator_.predict(
            X_test
        )

        results.append({

            "Dataset":
                dataset_name,

            "Model":
                model_name,

            "Accuracy":
                accuracy_score(
                    y_test,
                    y_pred
                ),

            "Precision":
                precision_score(
                    y_test,
                    y_pred,
                    average="macro"
                ),

            "Recall":
                recall_score(
                    y_test,
                    y_pred,
                    average="macro"
                ),

            "F1":
                f1_score(
                    y_test,
                    y_pred,
                    average="macro"
                )

        })

# =========================================================
# BEST HYPERPARAMETERS TABLE
# =========================================================

best_params_df = pd.DataFrame(
    best_parameters
)

best_params_df.to_csv(

    "Best_Hyperparameters.csv",

    index=False,

    encoding="utf-8-sig"

)

print("\nBEST PARAMETERS SAVED")

# =========================================================
# PERFORMANCE TABLE
# =========================================================

results_df = pd.DataFrame(
    results
)

results_df = results_df.sort_values(

    by="F1",

    ascending=False

)

print("\nMODEL PERFORMANCE")
print("=================================================")

display(results_df)

results_df.to_csv(

    "Model_Performance.csv",

    index=False,

    encoding="utf-8-sig"

)

# =========================================================
# SAVE ALL MODELS
# =========================================================

print("\nSAVING MODELS")
print("=================================================")

for name, model in best_models.items():

    filename = (

        name
        .replace(" ", "_")

        + "_best.pkl"
    )

    joblib.dump(
        model,
        filename
    )

    print(
        f"Saved: {filename}"
    )

# =========================================================
# BEST MODEL OVERALL
# =========================================================

best_model = results_df.iloc[0]

print("\nBEST MODEL OVERALL")
print("=================================================")

print(
    f"Dataset : {best_model['Dataset']}"
)

print(
    f"Model   : {best_model['Model']}"
)

print(
    f"F1 Score: {best_model['F1']:.4f}"
)

In [ ]:
# =========================================================
# IMPORTS
# =========================================================

import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier

# =========================================================
# DATASETS
# =========================================================

datasets = {

    "Original": (
        X_train,
        y_train
    ),

    "SMOTE": (
        X_train_smote,
        y_train_smote
    )
}

# =========================================================
# BEST PARAMETERS FROM GRIDSEARCH
# =========================================================

best_models_config = {

    "Original": {

        "Random Forest":
            RandomForestClassifier(
                n_estimators=300,
                max_depth=20,
                max_features="sqrt",
                min_samples_split=2,
                min_samples_leaf=1,
                random_state=42,
                n_jobs=-1
            ),

        "SVM":
            Pipeline([
                ("scaler", StandardScaler()),
                ("model",
                 SVC(
                     C=100,
                     gamma=1,
                     kernel="rbf",
                     probability=True,
                     random_state=42
                 ))
            ]),

        "kNN":
            Pipeline([
                ("scaler", StandardScaler()),
                ("model",
                 KNeighborsClassifier(
                     n_neighbors=3,
                     metric="manhattan",
                     weights="distance"
                 ))
            ]),

        "XGBoost":
            XGBClassifier(
                objective="multi:softprob",
                num_class=5,
                eval_metric="mlogloss",
                random_state=42,
                tree_method="hist",
                n_estimators=500,
                max_depth=10,
                learning_rate=0.1,
                subsample=1.0,
                colsample_bytree=1.0,
                min_child_weight=1
            )
    },

    "SMOTE": {

        "Random Forest":
            RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                max_features="sqrt",
                min_samples_split=2,
                min_samples_leaf=1,
                random_state=42,
                n_jobs=-1
            ),

        "SVM":
            Pipeline([
                ("scaler", StandardScaler()),
                ("model",
                 SVC(
                     C=100,
                     gamma=1,
                     kernel="rbf",
                     probability=True,
                     random_state=42
                 ))
            ]),

        "kNN":
            Pipeline([
                ("scaler", StandardScaler()),
                ("model",
                 KNeighborsClassifier(
                     n_neighbors=3,
                     metric="manhattan",
                     weights="distance"
                 ))
            ]),

        "XGBoost":
            XGBClassifier(
                objective="multi:softprob",
                num_class=5,
                eval_metric="mlogloss",
                random_state=42,
                tree_method="hist",
                n_estimators=500,
                max_depth=10,
                learning_rate=0.1,
                subsample=1.0,
                colsample_bytree=1.0,
                min_child_weight=1
            )
    }
}

# =========================================================
# TRAINING
# =========================================================

results = []
trained_models = {}

for dataset_name, (X_tr, y_tr) in datasets.items():

    print("\n")
    print("=" * 70)
    print(f"TRAINING USING {dataset_name.upper()} DATA")
    print("=" * 70)

    for model_name, model in best_models_config[dataset_name].items():

        print(f"\nTraining {model_name}")

        model.fit(X_tr, y_tr)

        trained_models[f"{model_name}_{dataset_name}"] = model

        y_pred = model.predict(X_test)

        results.append({

            "Dataset": dataset_name,
            "Model": model_name,

            "Accuracy":
                accuracy_score(
                    y_test,
                    y_pred
                ),

            "Precision":
                precision_score(
                    y_test,
                    y_pred,
                    average="macro"
                ),

            "Recall":
                recall_score(
                    y_test,
                    y_pred,
                    average="macro"
                ),

            "F1":
                f1_score(
                    y_test,
                    y_pred,
                    average="macro"
                )
        })

# =========================================================
# RESULTS
# =========================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1",
    ascending=False
)

print("\nMODEL PERFORMANCE")
print("=" * 70)

display(results_df)

results_df.to_csv(
    "Model_Performance_BestModels.csv",
    index=False,
    encoding="utf-8-sig"
)

# =========================================================
# SAVE MODELS
# =========================================================

print("\nSAVING MODELS")
print("=" * 70)

for name, model in trained_models.items():

    filename = (
        name.replace(" ", "_")
        + "_best.pkl"
    )

    joblib.dump(
        model,
        filename
    )

    print(f"Saved: {filename}")

# =========================================================
# BEST MODEL
# =========================================================

best_model = results_df.iloc[0]

print("\nBEST MODEL OVERALL")
print("=" * 70)

print(f"Dataset : {best_model['Dataset']}")
print(f"Model   : {best_model['Model']}")
print(f"F1 Score: {best_model['F1']:.4f}")

In [ ]:
# =========================================================
# GENERATE PREDICTIONS
# =========================================================

predictions_original = {

    "Random Forest":
        trained_models[
            "Random Forest_Original"
        ].predict(X_test),

    "SVM":
        trained_models[
            "SVM_Original"
        ].predict(X_test),

    "kNN":
        trained_models[
            "kNN_Original"
        ].predict(X_test),

    "XGBoost":
        trained_models[
            "XGBoost_Original"
        ].predict(X_test)

}

predictions_smote = {

    "Random Forest":
        trained_models[
            "Random Forest_SMOTE"
        ].predict(X_test),

    "SVM":
        trained_models[
            "SVM_SMOTE"
        ].predict(X_test),

    "kNN":
        trained_models[
            "kNN_SMOTE"
        ].predict(X_test),

    "XGBoost":
        trained_models[
            "XGBoost_SMOTE"
        ].predict(X_test)

}